In [18]:
import pandas as pd

# Replace 'path_to_your_file.csv' with the actual file path
df = pd.read_csv('..\data\captone_data.csv')

In [20]:
# --- CODE BLOCK 1 (Expanded): In-Depth EDA ---

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure to replace 'your_data_file.csv' with the actual name of your file
try:
    # --- Section 1: Basic Statistics (Recap) ---
    print("--- 1. Basic Statistics ---")
    num_users = df['user'].nunique()
    num_items = df['item'].nunique()
    print(f"Number of unique users: {num_users}")
    print(f"Number of unique items: {num_items}")
    print(f"Total number of ratings: {len(df)}")
    
    # Calculate sparsity before pivoting, which is more memory-efficient
    possible_ratings = num_users * num_items
    sparsity = 1 - (len(df) / possible_ratings)
    print(f"Sparsity of the dataset: {sparsity * 100:.2f}%")
    print(f"This means {sparsity*100:.2f}% of the user-item interactions are missing.")

    print("\n--- 2. Rating Distribution ---")
    print(f"Rating value counts:\n{df['rating'].value_counts().sort_index()}")
    print(f"\nAverage rating: {df['rating'].mean():.2f}")
    
    # --- Section 3: User Activity Analysis (Deeper Dive) ---
    print("\n--- 3. User Activity Analysis ---")
    user_counts = df['user'].value_counts()
    print("Descriptive statistics for number of ratings per user:")
    print(user_counts.describe())
    
    # Quantify the "long tail" for users
    print("\nUser long-tail distribution:")
    print(f"Number of users with only 1 rating: {sum(user_counts == 1)}")
    print(f"Number of users with 5 or fewer ratings: {sum(user_counts <= 5)}")
    print(f"% of users with 5 or fewer ratings: {100 * sum(user_counts <= 5) / num_users:.2f}%")


    # --- Section 4: Item Popularity Analysis (Deeper Dive) ---
    print("\n--- 4. Item Popularity Analysis ---")
    item_counts = df['item'].value_counts()
    print("Descriptive statistics for number of ratings per item:")
    print(item_counts.describe())

    # Quantify the "long tail" for items
    print("\nItem long-tail distribution:")
    print(f"Number of items with only 1 rating: {sum(item_counts == 1)}")
    print(f"Number of items with 5 or fewer ratings: {sum(item_counts <= 5)}")
    print(f"% of items with 5 or fewer ratings: {100 * sum(item_counts <= 5) / num_items:.2f}%")

    # This is a key finding for your presentation!
    print("\n--- 5. Cold Start Problem Identification ---")
    print("The 'cold start' problem refers to difficulty making recommendations for new users or items.")
    print(f"Our dataset contains {sum(user_counts <= 5)} users and {sum(item_counts <= 5)} items with 5 or fewer ratings.")
    print("These users/items are challenging for collaborative filtering.")


except FileNotFoundError:
    print("File not found. Please make sure your data file is in the correct directory and you've updated the filename in the code.")
except Exception as e:
    print(f"An error occurred: {e}")

# Note: This code will also generate plots. You should examine them, but only need to paste the text output.
# You can customize the plots to explore further if you wish.

--- 1. Basic Statistics ---
Number of unique users: 33901
Number of unique items: 126
Total number of ratings: 233306
Sparsity of the dataset: 94.54%
This means 94.54% of the user-item interactions are missing.

--- 2. Rating Distribution ---
Rating value counts:
2.0     10976
3.0    222330
Name: rating, dtype: int64

Average rating: 2.95

--- 3. User Activity Analysis ---
Descriptive statistics for number of ratings per user:
count    33901.000000
mean         6.881980
std          5.823548
min          1.000000
25%          2.000000
50%          6.000000
75%          9.000000
max         61.000000
Name: user, dtype: float64

User long-tail distribution:
Number of users with only 1 rating: 8320
Number of users with 5 or fewer ratings: 15018
% of users with 5 or fewer ratings: 44.30%

--- 4. Item Popularity Analysis ---
Descriptive statistics for number of ratings per item:
count      126.000000
mean      1851.634921
std       2874.579272
min          1.000000
25%        212.250000
50%

In [21]:
# --- CODE BLOCK 2: Pre-processing and Feature Engineering ---

import pandas as pd
from sklearn.model_selection import train_test_split

try:
    # --- 1. Load Data ---
    print("--- Initial Data Shape ---")
    print(f"Original shape: {df.shape}")
    num_users_orig = df['user'].nunique()
    num_items_orig = df['item'].nunique()
    print(f"Original unique users: {num_users_orig}")
    print(f"Original unique items: {num_items_orig}")

    # --- 2. Filter Data ---
    # Based on our EDA, we know many users have few ratings.
    # Let's set a threshold. A common choice is 5 or 10. Let's start with 5.
    min_ratings_per_user = 5
    
    user_counts = df['user'].value_counts()
    active_users = user_counts[user_counts >= min_ratings_per_user].index
    
    df_filtered = df[df['user'].isin(active_users)]
    
    print(f"\n--- Filtered Data Shape (Users with >= {min_ratings_per_user} ratings) ---")
    print(f"Filtered shape: {df_filtered.shape}")
    num_users_filtered = df_filtered['user'].nunique()
    num_items_filtered = df_filtered['item'].nunique()
    print(f"Filtered unique users: {num_users_filtered} ({(num_users_filtered / num_users_orig * 100):.2f}% of original)")
    print(f"Filtered unique items: {num_items_filtered} ({(num_items_filtered / num_items_orig * 100):.2f}% of original)")
    print(f"Number of ratings removed: {len(df) - len(df_filtered)}")
    
    # --- 3. Create User-Item Matrix & Split Data ---
    # IMPORTANT: We split the data BEFORE creating the matrix to prevent data leakage.
    # The test set should contain user-item pairs the model has never seen.
    
    train_df, test_df = train_test_split(df_filtered, test_size=0.2, random_state=42)
    
    print("\n--- Data Split ---")
    print(f"Training data size: {len(train_df)}")
    print(f"Test data size: {len(test_df)}")
    
    # Now, create the user-item matrix from the TRAINING data only.
    train_user_item_matrix = train_df.pivot_table(index='user', columns='item', values='rating')
    
    print("\n--- Final Training Matrix ---")
    print(f"Shape of the training user-item matrix: {train_user_item_matrix.shape}")
    
    # Let's re-calculate sparsity on the new matrix to see the improvement
    new_sparsity = train_user_item_matrix.isnull().sum().sum() / (train_user_item_matrix.shape[0] * train_user_item_matrix.shape[1])
    print(f"Sparsity of the new training matrix: {new_sparsity*100:.2f}%")


except FileNotFoundError:
    print("File not found. Please make sure your data file is in the correct directory and you've updated the filename in the code.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Initial Data Shape ---
Original shape: (233306, 3)
Original unique users: 33901
Original unique items: 126

--- Filtered Data Shape (Users with >= 5 ratings) ---
Filtered shape: (223776, 3)
Filtered unique users: 25062 (73.93% of original)
Filtered unique items: 124 (98.41% of original)
Number of ratings removed: 9530

--- Data Split ---
Training data size: 179020
Test data size: 44756

--- Final Training Matrix ---
Shape of the training user-item matrix: (25060, 122)
Sparsity of the new training matrix: 94.14%


In [22]:
# --- CODE BLOCK 3: Calculate User Similarity ---

import pandas as pd
# Assuming 'train_user_item_matrix' is in your environment from the previous step.
# If not, you'll need to re-run Code Block 2.

from sklearn.metrics.pairwise import cosine_similarity

try:
    print("--- Calculating User Similarity ---")
    print(f"Input training matrix shape: {train_user_item_matrix.shape}")

    # The user-item matrix has NaNs. Cosine similarity can't handle them.
    # We fill NaNs with 0. This means "no rating" is treated as 0.
    train_matrix_filled = train_user_item_matrix.fillna(0)
    print("\nFilled matrix with 0s to handle NaNs.")

    # Calculate cosine similarity between users (rows).
    # This is the most computationally intensive step. It may take a few moments.
    print("Calculating cosine similarity... (this might take a moment)")
    user_similarity = cosine_similarity(train_matrix_filled)
    print("Calculation complete.")

    # Wrap the resulting numpy array in a pandas DataFrame for easy lookup.
    # The index and columns will be the user IDs.
    user_similarity_df = pd.DataFrame(user_similarity,
                                      index=train_user_item_matrix.index,
                                      columns=train_user_item_matrix.index)

    print("\n--- User Similarity Matrix ---")
    print(f"Shape of the user similarity matrix: {user_similarity_df.shape}")
    print("\nExample of the similarity matrix (top 5x5 users):")
    print(user_similarity_df.iloc[:5, :5])

    # Let's test it by finding the most similar users to a random user.
    # Pick a user from the index.
    random_user = user_similarity_df.index[10] 
    print(f"\n--- Example: Finding neighbors for user '{random_user}' ---")
    
    # Get the similarity scores for this user, drop the user itself (similarity=1.0)
    similar_users = user_similarity_df[random_user].drop(random_user).sort_values(ascending=False)
    
    print("Top 5 most similar users:")
    print(similar_users.head())

except NameError:
    print("Error: 'train_user_item_matrix' not found. Please run the previous code block first.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Calculating User Similarity ---
Input training matrix shape: (25060, 122)

Filled matrix with 0s to handle NaNs.
Calculating cosine similarity... (this might take a moment)
Calculation complete.

--- User Similarity Matrix ---
Shape of the user similarity matrix: (25060, 25060)

Example of the similarity matrix (top 5x5 users):
user        2         4         5         9         12
user                                                  
2     1.000000  0.430018  0.375345  0.278735  0.367912
4     0.430018  1.000000  0.243757  0.430820  0.290129
5     0.375345  0.243757  1.000000  0.148522  0.233380
9     0.278735  0.430820  0.148522  1.000000  0.235702
12    0.367912  0.290129  0.233380  0.235702  1.000000

--- Example: Finding neighbors for user '30' ---
Top 5 most similar users:
user
480870     0.547723
686239     0.516398
1146994    0.516398
487965     0.507093
503365     0.507093
Name: 30, dtype: float64


In [25]:
# --- CODE BLOCK 4 (Revised): Prediction and Evaluation with F1-Score ---

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report

# Assumes these DataFrames are in your environment from previous steps:
# - train_df, test_df
# - train_user_item_matrix
# - user_similarity_df

try:
    # --- 1. Define the Prediction Function (MODIFIED for classification) ---
    def predict_class(user_id, item_id, k=40, threshold=2.5):
        # The core prediction logic remains the same (calculating the weighted float)
        # We'll re-use the function from before to get the float value first.
        # It's good practice to not duplicate code.
        
        # Inner function for float prediction (same as before)
        def predict_float_rating(u, i, k_inner):
            if u not in user_similarity_df.index or i not in train_user_item_matrix.columns:
                return train_df['rating'].mean()
            
            user_similarities = user_similarity_df[u]
            item_ratings = train_user_item_matrix[i]
            
            valid_indices = item_ratings.notna() & (user_similarities.index != u)
            neighbors = user_similarities[valid_indices]
            neighbor_ratings = item_ratings[valid_indices]
            
            if neighbors.empty:
                return train_df['rating'].mean()
            
            top_k_neighbors = neighbors.nlargest(k_inner)
            top_k_ratings = neighbor_ratings.loc[top_k_neighbors.index]
            
            if top_k_neighbors.empty:
                return train_df['rating'].mean()
                
            weighted_sum = (top_k_neighbors * top_k_ratings).sum()
            sum_of_weights = top_k_neighbors.sum()
            
            if sum_of_weights == 0:
                return train_df['rating'].mean()
                
            return weighted_sum / sum_of_weights
            
        # Get the predicted float rating
        predicted_float = predict_float_rating(user_id, item_id, k_inner=k)
        
        # Convert the float to a class based on the threshold
        if predicted_float > threshold:
            return 3.0
        else:
            return 2.0

    # --- 2. Evaluate on the entire test set ---
    print("\n--- Evaluating on the full test set for classification... (This may take a few minutes) ---")
    
    # Get the true labels from the test set
    y_true = test_df['rating']
    
    # Get the predicted labels (classes)
    y_pred = test_df.apply(
        lambda row: predict_class(row['user'], row['item'], k=40),
        axis=1
    )
    
    # --- 3. Calculate Final Performance Metrics ---
    # The primary metric is now F1-Score.
    # We specify pos_label=3.0 to get the F1 for the 'positive' class.
    f1 = f1_score(y_true, y_pred, pos_label=3.0)
    
    print("\n--- Final Model Performance (Classification) ---")
    print(f"The F1-Score of the model (for class 3.0) is: {f1:.4f}")
    
    # The classification_report gives a more detailed breakdown (very useful!)
    print("\n--- Detailed Classification Report ---")
    print(classification_report(y_true, y_pred))

except NameError as e:
    print(f"Error: A required DataFrame was not found. Please run the previous code blocks first. Details: {e}")
except Exception as e:
    print(f"An error occurred: {e}")


--- Evaluating on the full test set for classification... (This may take a few minutes) ---

--- Final Model Performance (Classification) ---
The F1-Score of the model (for class 3.0) is: 0.9973

--- Detailed Classification Report ---
              precision    recall  f1-score   support

         2.0       0.99      0.23      0.38       311
         3.0       0.99      1.00      1.00     44445

    accuracy                           0.99     44756
   macro avg       0.99      0.62      0.69     44756
weighted avg       0.99      0.99      0.99     44756

